In [ ]:
import sys
sys.dont_write_bytecode = True
sys.path.insert(0, '..')
from _init_notebook import *
sys.dont_write_bytecode = False

import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.dummy import DummyClassifier

# BTC Direction Predictor — ML Walk-Forward

**Objetivo:** predecir si BTC sube o baja al día siguiente usando el estado del mercado de hoy.

**Enfoque:**
- Features: régimen de tendencia de BTC + señales macro (VIX, S&P, DXY, Gold, Oil)
- Target: dirección de BTC al día siguiente (1 = sube, 0 = baja)
- Validación: walk-forward (sin data leakage)
- Modelo base: LightGBM

**Cambios v2 (fixes principales):**
- Eliminado `btc_ret_1d` como feature — causaba que el modelo aprendiera mean-reversion (BTC subió hoy → predigo baja mañana), que es destructivo en un activo que trendea
- Agregados features de régimen de tendencia: distancia a MA50/200, posición relativa
- Agregados VIX spikes multiperiodo
- Walk-forward con filtro de confianza: solo operar cuando `proba > umbral`

**Baseline a superar:** predecir siempre la clase mayoritaria (~54% en BTC)

## 1. Cargar datos

In [ ]:
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'yahoo_1d')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')

def load_close(group, symbol):
    """Carga la serie de cierre de un símbolo."""
    fname = symbol.replace('^', '').replace('/', '-')
    path = os.path.join(DATA_DIR, group, f'{fname}.csv')
    if not os.path.exists(path):
        print(f'No encontrado: {path}')
        return None
    df = pd.read_csv(path, skiprows=[1])
    df.columns = ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.set_index('Date').sort_index()
    return df['Close'].rename(symbol)

# BTC (target)
btc = load_close('cripto', 'BTC-USD')

# Macro features
vix   = load_close('indexes', '^VIX')
sp500 = load_close('indexes', '^GSPC')
dxy   = load_close('indexes', 'DX-Y.NYB')
gold  = load_close('commodities', 'GC=F')
oil   = load_close('commodities', 'CL=F')

# Combinar — inner join para quedarnos con días hábiles donde todos tienen datos
raw = pd.concat([btc, vix, sp500, dxy, gold, oil], axis=1, join='inner').dropna()
print(f'Datos combinados: {len(raw)} filas | {raw.index.min().date()} → {raw.index.max().date()}')
raw.tail()

## 2. Feature engineering

In [ ]:
df = raw.copy()
btc_ret = df['BTC-USD'].pct_change()  # guardamos para equity curve, NO como feature

# --- Target: dirección de BTC mañana ---
df['target'] = (df['BTC-USD'].shift(-1) > df['BTC-USD']).astype(int)

# --- BTC: momentum multiperiodo (sin el retorno de 1 día — evita mean-reversion) ---
for w in [3, 7, 14, 30]:
    df[f'btc_ret_{w}d']  = df['BTC-USD'].pct_change(w)
    df[f'btc_vol_{w}d']  = btc_ret.rolling(w).std()

# --- BTC: régimen de tendencia (MA) ---
for w in [20, 50, 200]:
    ma = df['BTC-USD'].rolling(w).mean()
    df[f'btc_dist_ma{w}'] = (df['BTC-USD'] - ma) / ma   # distancia % a la MA
df['btc_above_200ma'] = (df['BTC-USD'] > df['BTC-USD'].rolling(200).mean()).astype(int)
df['btc_ma50_above_200'] = (df['BTC-USD'].rolling(50).mean() > df['BTC-USD'].rolling(200).mean()).astype(int)  # golden/death cross

# --- BTC: indicadores técnicos ---
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands
from ta.trend import MACD

df['btc_rsi_14']   = RSIIndicator(df['BTC-USD'], window=14).rsi()
df['btc_rsi_7']    = RSIIndicator(df['BTC-USD'], window=7).rsi()
bb = BollingerBands(df['BTC-USD'], window=20)
df['btc_bb_pct']   = bb.bollinger_pband()   # posición dentro de las bandas (0=low, 1=high)
df['btc_bb_width'] = bb.bollinger_wband()   # ancho de bandas = régimen de volatilidad
df['btc_macd_diff'] = MACD(df['BTC-USD']).macd_diff()

# --- Macro: retornos diarios y momentum ---
for col, group_name in [('^VIX', 'vix'), ('^GSPC', 'sp500'), ('DX-Y.NYB', 'dxy'), ('GC=F', 'gold'), ('CL=F', 'oil')]:
    df[f'{group_name}_ret_1d'] = df[col].pct_change()
    df[f'{group_name}_ret_5d'] = df[col].pct_change(5)

# VIX: nivel y spikes (señales de pánico)
df['vix_level']    = df['^VIX']
df['vix_spike_3d'] = df['^VIX'].pct_change(3)   # subida brusca del VIX en 3 días
df['vix_spike_7d'] = df['^VIX'].pct_change(7)

# Correlación rolling BTC vs S&P (régimen de acoplamiento al mercado tradicional)
df['btc_sp500_corr_30d'] = btc_ret.rolling(30).corr(df['^GSPC'].pct_change())

# Limpiar
df['btc_ret_1d'] = btc_ret  # lo guardamos solo para la equity curve, no entra en features
df = df.dropna()

FEATURE_COLS = [c for c in df.columns if c not in [
    'target', 'btc_ret_1d',                          # excluidos explícitamente
    'BTC-USD', '^VIX', '^GSPC', 'DX-Y.NYB', 'GC=F', 'CL=F'   # precios raw
]]
print(f'Features ({len(FEATURE_COLS)}):')
for f in FEATURE_COLS:
    print(f'  {f}')
print(f'\nFilas con features completas: {len(df)}')
print(f'Balance de clases — sube: {df["target"].mean():.1%} | baja: {1 - df["target"].mean():.1%}')

## 3. Walk-forward validation

Entrenamos con datos históricos y predecimos el período siguiente, avanzando en el tiempo.
Esto evita data leakage y simula condiciones reales de trading.

In [ ]:
TRAIN_YEARS = 3
TEST_MONTHS = 6
CONFIDENCE_THRESHOLD = 0.55  # solo operar cuando el modelo está seguro

X = df[FEATURE_COLS]
y = df['target']
dates = df.index

all_probas = []
all_true   = []
all_dates  = []
fold_results = []

start_date = dates[0] + pd.DateOffset(years=TRAIN_YEARS)
current    = start_date
end_date   = dates[-1]

fold = 0
while current < end_date:
    test_end = current + pd.DateOffset(months=TEST_MONTHS)
    train_mask = dates < current
    test_mask  = (dates >= current) & (dates < test_end)

    if test_mask.sum() < 10:
        break

    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]

    model = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        verbose=-1,
        random_state=42
    )
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba > 0.5).astype(int)

    # Accuracy con y sin filtro de confianza
    confident_mask = (proba > CONFIDENCE_THRESHOLD) | (proba < (1 - CONFIDENCE_THRESHOLD))
    acc_all        = accuracy_score(y_test, preds)
    acc_confident  = accuracy_score(y_test[confident_mask], preds[confident_mask]) if confident_mask.sum() > 0 else np.nan
    auc            = roc_auc_score(y_test, proba)

    fold_results.append({
        'fold': fold, 'start': current.date(), 'end': test_end.date(),
        'n_train': train_mask.sum(), 'n_test': test_mask.sum(),
        'n_confident': confident_mask.sum(),
        'acc_all': acc_all, 'acc_confident': acc_confident, 'auc': auc
    })

    all_probas.extend(proba)
    all_true.extend(y_test)
    all_dates.extend(dates[test_mask])

    current = test_end
    fold += 1

folds_df = pd.DataFrame(fold_results)
print(folds_df[['fold','start','end','n_train','n_test','n_confident','acc_all','acc_confident','auc']].to_string(index=False))
print(f"\nAccuracy promedio (todos):        {folds_df['acc_all'].mean():.3f}")
print(f"Accuracy promedio (confianza>{CONFIDENCE_THRESHOLD}): {folds_df['acc_confident'].mean():.3f}")
print(f"AUC promedio:                     {folds_df['auc'].mean():.3f}")
print(f"% días en que opera:              {folds_df['n_confident'].sum() / folds_df['n_test'].sum():.1%}")

In [ ]:
# Baseline: predecir siempre la clase mayoritaria
dummy = DummyClassifier(strategy='most_frequent')

# Aplicar el mismo walk-forward al baseline
dummy_accs = []
current = start_date
while current < end_date:
    test_end = current + pd.DateOffset(months=TEST_MONTHS)
    train_mask = dates < current
    test_mask  = (dates >= current) & (dates < test_end)
    if test_mask.sum() < 10:
        break
    dummy.fit(X[train_mask], y[train_mask])
    dummy_accs.append(accuracy_score(y[test_mask], dummy.predict(X[test_mask])))
    current = test_end

print(f"Baseline (clase mayoritaria): {np.mean(dummy_accs):.3f}")
print(f"Nuestro modelo:               {folds_df['accuracy'].mean():.3f}")
print(f"Mejora sobre baseline:        {folds_df['accuracy'].mean() - np.mean(dummy_accs):+.3f}")

## 4. Feature importance

In [ ]:
# Entrenar un modelo final con todos los datos para ver importancia de features
model_final = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05,
    num_leaves=15, min_child_samples=20,
    verbose=-1, random_state=42
)
model_final.fit(X, y)

fi = pd.Series(model_final.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, len(FEATURE_COLS) * 0.4))
fi.plot.barh(ax=ax, color='steelblue')
ax.set_title('Feature importance (LightGBM)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'btc_feature_importance.png'), dpi=150)
plt.show()

## 5. Equity curve simulada

Simulación naive: entrar long cuando el modelo predice subida, cash cuando predice bajada.

In [ ]:
pred_df = pd.DataFrame({
    'proba': all_probas,
    'actual': all_true,
    'btc_ret': df.loc[all_dates, 'btc_ret_1d'].values
}, index=all_dates)

pred_df['pred']       = (pred_df['proba'] > 0.5).astype(int)
pred_df['confident']  = (pred_df['proba'] > CONFIDENCE_THRESHOLD) | (pred_df['proba'] < (1 - CONFIDENCE_THRESHOLD))

# Estrategias
pred_df['ret_bh']         = pred_df['btc_ret']
pred_df['ret_ml_all']     = pred_df['btc_ret'] * pred_df['pred']                             # opera todos los días
pred_df['ret_ml_conf']    = pred_df['btc_ret'] * pred_df['pred'] * pred_df['confident']      # solo días de alta confianza

equity_bh      = (1 + pred_df['ret_bh']).cumprod()
equity_ml_all  = (1 + pred_df['ret_ml_all']).cumprod()
equity_ml_conf = (1 + pred_df['ret_ml_conf']).cumprod()

fig, ax = plt.subplots(figsize=(13, 5))
equity_bh.plot(ax=ax,      label='Buy & Hold BTC',                     color='gray',      alpha=0.6)
equity_ml_all.plot(ax=ax,  label='ML (todos los días)',                 color='tomato',    alpha=0.8)
equity_ml_conf.plot(ax=ax, label=f'ML (confianza > {CONFIDENCE_THRESHOLD})', color='steelblue')
ax.set_title('Equity curve — Walk-forward (sin costos de transacción)')
ax.set_ylabel('Capital (inicio = 1)')
ax.axhline(1, color='black', linewidth=0.8, linestyle='--')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'btc_equity_curve.png'), dpi=150)
plt.show()

print(f"Buy & Hold:               {equity_bh.iloc[-1] - 1:+.1%}")
print(f"ML todos los días:        {equity_ml_all.iloc[-1] - 1:+.1%}")
print(f"ML confianza>{CONFIDENCE_THRESHOLD}:         {equity_ml_conf.iloc[-1] - 1:+.1%}")